# Coding Model Architecture

Under the hood, most coding models are decoder-only transformers with code-centric pretraining objectives, specialized tokenizers, long-context strategies, and a post-training stack (SFT + preference/RL). This notebook builds architectural intuition useful for model selection and system design.

```mermaid
flowchart TB
  Data[Code + Text corpora] --> Pre[Pretraining]
  Pre --> Obj[NTP / FIM / Edit]
  Obj --> Base[Base coding LM]
  Base --> SFT[SFT on instructions]
  SFT --> Pref[Preference / RL / tools]
  Pref --> Serve[Serving + product adapters]
```


## Learning Objectives

- Contrast decoder-only vs encoder-decoder for code
- Explain pretraining objectives used for code models
- Budget tokens and understand tokenization quirks
- Describe long-context and position-encoding tradeoffs
- Outline the post-training stack for assistants/agents


## 1. Base Architecture Patterns

### Why decoder-only dominates
- Unified next-token interface for completion and chat
- Scales well with data/compute
- Easy to adapt with SFT and tool-calling formats

### Encoder-decoder (historical / niche)
Useful for translation-like tasks (code→code, code→doc) but less common for interactive IDE products.

| Pattern | Pros | Cons | When |
|---------|------|------|------|
| Decoder-only | Flexible, standard tooling | Full attention cost | Default choice |
| Encoder-decoder | Natural for transduction | Two stacks to train/serve | Specialized translators |
| Diffusion / other | Research | Immature tooling | Experimental |

### Pitfalls
Assuming architecture alone determines coding skill—**data and objectives** usually dominate.


## 2. Pretraining Objectives for Code

| Objective | Definition | Why for code |
|-----------|------------|--------------|
| Next-token prediction | Predict token t+1 | General fluency |
| FIM | Predict middle span | IDE holes |
| Next-edit / diff | Predict edit given before | Inline edits |
| Repo packing | Pack related files | Cross-file priors |

### Intuition
Code is highly structured and often edited in the middle of files. FIM and edit prediction align training with product UX.


In [ ]:
# Demo 1 — Construct a simple FIM training sample
import random

def make_fim_sample(code: str, min_mid: int = 10, max_mid: int = 40) -> dict:
    if len(code) < min_mid + 20:
        raise ValueError("code too short")
    mid_len = random.randint(min_mid, min(max_mid, len(code)//3))
    start = random.randint(0, len(code) - mid_len)
    prefix, middle, suffix = code[:start], code[start:start+mid_len], code[start+mid_len:]
    return {
        "prefix": prefix,
        "middle": middle,
        "suffix": suffix,
        "train_text": f"<fim_prefix>{prefix}<fim_suffix>{suffix}<fim_middle>{middle}",
    }

src = "def greet(name):\n    return f'hi {name}'\n\ndef main():\n    print(greet('a'))\n"
random.seed(0)
print(make_fim_sample(src)["train_text"][:120], "...")


In [ ]:
# Demo 2 — Diff / next-edit style sample
def make_edit_sample(before: str, after: str) -> str:
    return (
        "### BEFORE\n" + before +
        "\n### AFTER\n" + after
    )

before = "def add(a,b):\n    return a-b\n"
after = "def add(a,b):\n    return a+b\n"
print(make_edit_sample(before, after))


## 3. Tokenization Considerations

### Why it matters
Tokenization affects:
- Cost (tokens ≈ dollars)
- Context fit (how much repo fits)
- Indentation/whitespace efficiency
- Multilingual code / unusual identifiers

### Pitfalls
- Estimating tokens with English word counts for code
- Ignoring that JSON/YAML can be surprisingly token-heavy


In [ ]:
# Demo 3 — Rough token estimators + indentation sensitivity
def estimate_tokens(text: str, chars_per_tok: float = 4.0) -> int:
    return max(1, int(len(text) / chars_per_tok))

samples = {
    "dense_code": "x=a+b\n" * 50,
    "verbose_json": '{\n  "user_id": "abc",\n  "roles": ["admin"]\n}\n' * 20,
    "indented": (" " * 4 + "if x:\n" + " " * 8 + "y()\n") * 30,
}
for k, v in samples.items():
    print(f"{k:14} chars={len(v):5} ~tok={estimate_tokens(v):5}")


## 4. Long Context & Position Encoding

### Techniques (conceptual)
- Absolute / relative positions
- ALiBi, RoPE scaling, YaRN-style methods
- Sliding windows / attention sinks
- Retrieval instead of raw long context

### When to use long context
- Whole-PR reasoning with many small files
- Log + code joint diagnosis

### When not to
- When retrieval can provide a sharper needle
- When latency/cost explode for little gain


In [ ]:
# Demo 4 — Context strategy chooser
def choose_context_strategy(n_files: int, avg_tokens: int, needle_known: bool) -> str:
    total = n_files * avg_tokens
    if needle_known:
        return "targeted_retrieval"
    if total < 20_000:
        return "pack_full_subset"
    if total < 120_000:
        return "retrieve_then_pack"
    return "agentic_search_tools"

for case in [(5, 800, False), (200, 900, False), (200, 900, True)]:
    print(case, "->", choose_context_strategy(*case))


## 5. Post-training Stack

1. **SFT** — instruction / chat / coding task format
2. **Preference optimization** — human or test-based preferences
3. **RL from verifiers** — unit tests, linters as reward
4. **Tool-use training** — trajectories with shell/search
5. **Safety / policy** — refuse malware, exfil, etc.

### Pitfalls
Over-aligning to chat style can hurt raw completion quality; products sometimes use different models/modes for inline vs chat.


### Try it yourself — Architecture

- Build 100 FIM samples from an open-source file and inspect span length distribution
- Compare token estimates vs tiktoken for the same code snippet
- Write a one-pager recommending context strategy for your monorepo


## Glossary / Key Terms

| Term | Meaning |
|------|--------|
| `NTP` | Next-token prediction |
| `SFT` | Supervised fine-tuning |
| `RoPE` | Rotary position embeddings |
| `Verifier` | Programmatic reward signal such as tests |


## Interview Prep — Sample Q&A

Practice answering out loud, then compare to the sample answers.


### Q1. Why is FIM important for coding models?

**Sample answer**

IDE edits often happen mid-file. FIM trains the model to condition on both sides of the cursor, improving inline completion quality versus pure left-to-right training.


## Deep Dive Workshop — 03 Coding Model Architecture

This section expands the notebook into instructor/textbook depth. Work through each subsection: **definition → why it matters → how it works → intuition → pitfalls → when to use**.

```mermaid
flowchart TB
  D[Definition] --> W[Why it matters]
  W --> H[How it works]
  H --> I[Intuition]
  I --> P[Pitfalls]
  P --> U[When to use]
```


### Concept card pack for `03-coding-model-architecture`

| Concept | Definition | Why it matters | Common pitfall |
|---------|------------|----------------|----------------|
| Primary abstraction | Core object this lesson centers on | Anchors design conversations | Vague naming |
| Quality oracle | How you know the system is right | Prevents demo-driven development | Using vibes only |
| Latency budget | Max user-visible wait | Drives architecture | Ignoring TTFT vs e2e |
| Cost unit | $ per successful task | Makes tradeoffs real | Optimizing tokens not outcomes |
| Trust boundary | Where data/control changes hands | Security design | Treating vendors as internal |
| Feedback loop | How production improves the system | Sustainable quality | No path from thumbs-down to evals |

**Intuition:** If you cannot fill this table for your system, you are not ready to choose models or frameworks.


### Pipeline walkthrough (apply to 03-coding-model-architecture)

```
1. Input arrives (user / job / webhook)
2. Normalize + authorize + budget check
3. Gather context (files, RAG, tools, memory)
4. Model / deterministic compute
5. Validate output (schema, policy, tests)
6. Side effects (write, ticket, PR) with authz
7. Observe (metrics, traces, feedback)
8. Learn (eval suite growth, prompt/model revision)
```

**When to compress steps:** tiny internal tools. **When to keep all steps:** multi-tenant or regulated production.


### Coding-models advanced notes

**Fill-in-the-middle formats** differ by vendor; always keep an adapter layer.  
**Repo agents** should treat tests as the north star and protect test files by default.  
**Coding RAG** should combine symbol lookup + BM25 + embeddings; embeddings alone miss identifiers.

| Task | Prefer | Avoid |
|------|--------|-------|
| Ghost text | FIM-capable small/fast model | Giant chat model sync |
| API migration | Agent + tests | Single-shot whole-repo rewrite |
| Explain legacy | Chat + citations | Uncited summaries |


In [ ]:
# Extra demo — diff extraction toy for coding assistants
import re

def extract_fenced_blocks(text: str) -> list[tuple[str, str]]:
    pat = re.compile(r"```(\w+)?\n(.*?)```", re.S)
    return [(m.group(1) or 'txt', m.group(2)) for m in pat.finditer(text)]

sample = '''Here is a fix:\n```python\ndef add(a,b):\n    return a+b\n```\n'''
print(extract_fenced_blocks(sample))


In [ ]:
# Extra demo — simple symbol index for coding RAG
import ast
from collections import defaultdict

def index_symbols(source: str, path: str) -> dict[str, list[str]]:
    tree = ast.parse(source)
    idx = defaultdict(list)
    for n in ast.walk(tree):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            idx[n.name].append(f"{path}:{n.lineno}")
    return dict(idx)

print(index_symbols('class Foo:\n  def bar(self):\n    pass\n', 'a.py'))


### Sample interview Q&A — coding models

**Q:** Copilot-quality inline completion is slow. What do you do?  
**A:** Separate completion model from chat model; shrink context to locals + imports; consider speculative decoding / smaller quantized model; measure acceptance rate not just tok/s.

**Q:** How do you evaluate a coding assistant for a monorepo?  
**A:** Private suite: completion acceptance, unit-test pass on generated patches, security scanner findings, and human review on a stratified sample of PR diffs.


### Comparison matrix exercise

Fill this for two competing designs in this topic:

| Dimension | Option A | Option B | Winner / why |
|-----------|----------|----------|--------------|
| Latency | | | |
| Cost at 10× scale | | | |
| Quality risk | | | |
| Ops burden | | | |
| Security / privacy | | | |
| Time to MVP | | | |


In [ ]:
# Workshop demo — decision scorecard
from dataclasses import dataclass

@dataclass
class Option:
    name: str
    latency: int  # 1=best .. 5=worst
    cost: int
    quality_risk: int
    ops: int
    security: int

def score(o: Option, weights=None) -> float:
    weights = weights or dict(latency=1, cost=1, quality_risk=2, ops=1, security=2)
    return (
        o.latency*weights['latency'] + o.cost*weights['cost'] +
        o.quality_risk*weights['quality_risk'] + o.ops*weights['ops'] +
        o.security*weights['security']
    )

a = Option('A', 2, 3, 2, 2, 2)
b = Option('B', 3, 1, 3, 4, 2)
print(a.name, score(a), b.name, score(b), '-> prefer', a.name if score(a)<score(b) else b.name)


In [ ]:
# Workshop demo — experiment log (use while studying this notebook)
from dataclasses import dataclass, asdict
import json, time

@dataclass
class Experiment:
    hypothesis: str
    setup: str
    metric: str
    baseline: float | None = None
    treatment: float | None = None
    notes: str = ''
    ts: float = 0.0

    def __post_init__(self):
        if not self.ts:
            self.ts = time.time()

exp = Experiment(
    hypothesis='Technique from this lesson improves the primary metric',
    setup='Describe fixtures / model / dataset version',
    metric='name of metric',
    baseline=0.0,
    treatment=0.0,
)
print(json.dumps(asdict(exp), indent=2))


### ASCII architecture sketch template

```
[ Clients ]
     |
[ Edge / API Gateway ] -- authn/z, rate limit
     |
[ Orchestration ] ------+-- prompts / policies
     |                  +-- eval hooks
     +-- context layer (RAG / tools / memory)
     |
[ Model interface ] ---- local and/or cloud
     |
[ Data plane ] --------- indexes, OLTP, object store
     |
[ Observability ] ------ logs, metrics, traces, feedback
```

Copy into your notes and annotate trust boundaries with `***`.


### Pitfalls clinic (read aloud)

1. **Metric theater** — optimizing a proxy that users don't feel  
2. **Context stuffing** — more tokens ≠ more truth  
3. **Prompt as security** — never the only control  
4. **Hidden coupling** — tools/models/indexes version-drift  
5. **No rollback** — can't revert prompt/model quickly  
6. **Eval contamination** — testing on training-like snippets  
7. **Happy-path demos** — skipping adversarial & empty-retrieve cases  


### Try it yourself — extended set

1. Teach the top 3 ideas from this notebook to a rubber duck in 5 minutes  
2. Write 5 quiz questions (with answers) for a junior engineer  
3. Implement one code demo with a real dependency (API or local model) using env placeholders  
4. Break a naive design on purpose; list the failure mode and the fix  
5. Add two rows to your personal glossary with examples from work  
6. Produce a one-page cheat sheet you could use in an interview  


### Mini case study

**Scenario:** Leadership wants this capability in production in six weeks with two engineers.

**Your job:** Propose an MVP that keeps irreversible risks controlled, names the eval gates, and lists what you explicitly defer.

Deliverable structure:
- MVP user story  
- Non-goals  
- Architecture (6 boxes max)  
- Eval gate table  
- Risk register (top 5)  
- Week-by-week plan  


In [ ]:
# Case study helper — risk register
import pandas as pd

risks = pd.DataFrame([
    {'risk': 'quality_miss', 'likelihood': 3, 'impact': 3, 'mitigation': 'golden evals + canary'},
    {'risk': 'cost_overrun', 'likelihood': 3, 'impact': 2, 'mitigation': 'budgets + cache'},
    {'risk': 'data_leak', 'likelihood': 2, 'impact': 5, 'mitigation': 'ACL + redaction'},
    {'risk': 'prompt_injection', 'likelihood': 4, 'impact': 4, 'mitigation': 'boundaries + allowlists'},
    {'risk': 'ops_pages', 'likelihood': 3, 'impact': 3, 'mitigation': 'runbooks + rollback'},
])
risks['score'] = risks.likelihood * risks.impact
print(risks.sort_values('score', ascending=False).to_string(index=False))


### Interview drill (topic-local)

Use the STAR or design template. Timebox 8 minutes.

**Prompt:** “Walk me through how you would productionize the main idea of this notebook.”

Checklist for a strong answer:
- [ ] Clarifying questions  
- [ ] Constraints & numbers  
- [ ] Diagram  
- [ ] Deep dive on hardest part  
- [ ] Evals  
- [ ] Security  
- [ ] Rollout / rollback  


### Glossary boost

| Term | Expanded meaning |
|------|------------------|
| Canary | Partial traffic to a new variant with automatic rollback |
| Golden set | Versioned labeled examples for regression |
| TTFT | Time to first token — interactive UX driver |
| Packing | Selecting/ordering context under a token budget |
| HITL | Human approval inserted before side effects |
| Idempotency | Safe retries without duplicate side effects |
| Shadow traffic | New system sees traffic but doesn't affect users |
| Circuit breaker | Stop calling a failing dependency temporarily |


In [ ]:
# Self-check quiz (run and answer mentally before printing answers)
QUESTIONS = [
    'What oracle proves success for this topic?',
    'Name one metric that can be gamed and a better alternative.',
    'What is the top security failure mode?',
    'What would you defer in an MVP?',
    'How do you rollback a bad change here?',
]
for i, q in enumerate(QUESTIONS, 1):
    print(f'Q{i}. {q}')
print('\n--- suggested answer hints ---')
HINTS = [
    'executable tests / task success / human rubric',
    'longer answers != better; use task success',
    'trust boundary crossing / injection / ACL',
    'multi-agent, perfect UI, every connector',
    'versioned prompts/models + traffic switch',
]
for h in HINTS:
    print('-', h)


### Further practice roadmap for `03-coding-model-architecture`

| Horizon | Action |
|---------|--------|
| Today | Re-run all code cells; note questions |
| This week | Apply one technique to a real repo/service |
| This month | Add an eval or security test covering this topic |
| Interview ready | Give a 10-minute teach-back with a diagram |


## Lab: End-to-end scenario

Work this scenario in your notes, then implement the smallest possible spike.

### Scenario brief
A team wants to adopt the techniques from this notebook for a **real internal tool** used daily by 200 people. Leadership cares about reliability and auditability more than flashy demos.

### Deliverables
1. One-paragraph problem statement  
2. Success metrics (3) with oracles  
3. Architecture sketch with trust boundaries  
4. Threats / failure modes (5)  
5. Eval plan (offline + online)  
6. 2-week MVP scope and explicit non-goals  

### Review questions
- What happens when context is empty?  
- What happens when the model is down?  
- What happens when a user is malicious?  
- How do you prove a release is safer/better than last week?  


In [ ]:
# Lab helper — MVP scope tracker
from dataclasses import dataclass, field

@dataclass
class MVP:
    must: list[str] = field(default_factory=list)
    should: list[str] = field(default_factory=list)
    defer: list[str] = field(default_factory=list)

    def show(self):
        for label, items in [('MUST', self.must), ('SHOULD', self.should), ('DEFER', self.defer)]:
            print(label)
            for i in items:
                print(' -', i)

mvp = MVP(
    must=['core happy path', 'authn', 'basic eval smoke', 'rollback switch'],
    should=['streaming UX', 'dashboards'],
    defer=['multi-agent', 'perfect personalization', 'every connector'],
)
mvp.show()


## Operator runbook sketch

| Symptom | Likely cause | First checks | Mitigation |
|---------|--------------|--------------|------------|
| Latency spike | Downstream model / retrieve | p95 by stage, saturation | shed load, failover |
| Quality drop | Prompt/model/index change | diff versions, eval slice | rollback |
| Cost spike | loops / huge prompts | tokens/req, step counts | budget breaker |
| Security alert | injection / ACL | traces + retrieved IDs | kill switch |

Keep this table in your ops wiki; customize per system.


In [ ]:
# Operator helper — stage latency rollup
from statistics import mean

stages = {
    'gateway': [20, 25, 22],
    'retrieve': [80, 120, 95],
    'generate': [900, 1100, 980],
}
for k, v in stages.items():
    print(f'{k:10} mean={mean(v):.0f}ms max={max(v)}ms')
print('e2e~', sum(mean(v) for v in stages.values()), 'ms')


## Teaching notes (for study groups)

- Start with the comparison table; argue both sides for 5 minutes  
- Pair-program one demo cell with a real endpoint (placeholder keys)  
- Each person writes one failure case the suite must catch  
- End with a 60-second summary of when *not* to use the technique  


## Summary & Key Takeaways

- Decoder-only transformers dominate coding assistants
- Objectives (FIM/edit/repo packing) align training with UX
- Tokenization drives cost and packing strategy
- Long context is a tool—not a substitute for retrieval
- Post-training shapes assistant/agent behavior
